Approach :

User Query

LLM -> Extract Structured Filters (JSON)

Python Pandas -> Filtering and Sorting

LLM -> Generate Final Answer.

In [26]:
import pandas as pd 

df = pd.read_csv("cleaned_data/bangalore_cleaned.csv")
df.shape

(7144, 9)

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7144 entries, 0 to 7143
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     7144 non-null   object 
 1   availability  7144 non-null   object 
 2   location      7144 non-null   object 
 3   size          7144 non-null   object 
 4   society       7144 non-null   object 
 5   total_sqft    7144 non-null   object 
 6   bath          7144 non-null   float64
 7   balcony       7144 non-null   float64
 8   price         7144 non-null   float64
dtypes: float64(3), object(6)
memory usage: 502.4+ KB


In [28]:
def convert_sqft(x):
    try:
        if '-' in str(x):
            a,b = x.split('-')
            return (float(a) + float(b))/2
        return float(x)
    except:
        return None 
    
df['total_sqft'] = df['total_sqft'].apply(convert_sqft)

# Dropping missing values in total_sqft after conversion
df = df.dropna(subset=['total_sqft'])


In [29]:
# extract bhk from size column

import re 

def extract_bhk(x):
    if pd.isna(x):
        return None
    x = x.lower().strip()

    match = re.search(r'(\d+)\s*(bhk|bedroom|rk)',x)

    if match:
        return int(match.group(1))
    
    return None

df['bhk'] = df['size'].apply(extract_bhk) 


# Dropping missing values in bhk after extraction
df = df.dropna(subset=['bhk'])

We will use the LLM for Filter Extraction

In [30]:
import os 
os.environ['HF_HOME'] = "D:/hf_cache"

In [31]:
from transformers import pipeline

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'

gen = pipeline(
    task = 'text-generation',
    model = model_id,
    device = 0,
    model_kwargs = {'cache_dir': "D:/hf_cache"}
)

d:\apps\InstalledApplications\anaconda3\envs\realestate_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 716.69it/s, Materializing param=model.norm.weight]                              


In [33]:
df.columns

Index(['area_type', 'availability', 'location', 'size', 'society',
       'total_sqft', 'bath', 'balcony', 'price', 'bhk'],
      dtype='object')

Testing for making the prompt better for filter extraction and improving the accuracy of the extracted filters.

In [ ]:
query = "Show me 2 BHK apartments in Whitefield under 80 lakhs sorted by price low to high"
prompt = f"""
    Extract filters from the real estate query.

    Return ONLY valid JSON with these fields:
    - bhk (int or null)
    - location (string or null)
    - max_price (float or null , in lakhs) 
    - min_price (float or null , in lakhs) 
    - min_sqft (float or null , in sqft)
    - max_sqft (float or null , in sqft) 
    - sort_by ("price" or "sqft")
    - sort_order ("asc" or "desc")


    Query: {query}
    """ 


out = gen(prompt, max_new_tokens=200, do_sample=False , max_length = None)

In [ ]:
print(out[0]['generated_text'])

In [ ]:
query1 = "Show me 3 BHK apartments in Whitefield under 3 crores sorted by sqft high to low"
prompt1 = f"""
    Extract filters from the real estate query.

    Return ONLY valid JSON with these fields:
    - bhk (int or null)
    - location (string or null)
    - max_price (float or null , in lakhs) 
    - min_price (float or null , in lakhs) 
    - min_sqft (float or null , in sqft)
    - max_sqft (float or null , in sqft) 
    - sort_by ("price" or "sqft")
    - sort_order ("asc" or "desc")


    Query: {query1}
    """ 


out1 = gen(prompt1, max_new_tokens=200, do_sample=False , max_length = None)

In [ ]:
print(out1[0]['generated_text'])

# Not able to extract the filters here.

We will Implement Few shot prompting for better accuracy of filter extraction.

In [61]:
query1 = "Show me 4-BHK apartments in Whitefield with price less than 1 crore"
prompt1 = f"""
    Extract filters from the real estate query.

    Return ONLY valid JSON with these fields:
    - bhk (int or null)
    - location (string or null)
    - max_price (float or null , in lakhs) 
    - min_price (float or null , in lakhs) 
    - min_sqft (float or null , in sqft)
    - max_sqft (float or null , in sqft) 
    - sort_by ("price" or "sqft")
    - sort_order ("asc" or "desc")
    
    INSTRUCTION:
     DO NOT RETURN ANY EXPLANATION. ONLY RETURN THE JSON.DO NOT RETURN ANY CODE FOR QUERY MANIPULATION TO JSON. ONLY RETURN THE JSON.
    
    Examples:
    Query:
    Show me 2 BHK apartments in Whitefield under 80 lakhs sorted by price low to high
    Output: {{"bhk": 2, "location": "Whitefield", "max_price": 80.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": "price", "sort_order": "asc"}}

    Query:
    Find 3 BHK flats in Koramangala under 1.5 crores
    Output: {{"bhk": 3, "location": "Koramangala", "max_price": 150.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null}}

    Query:
    Show me apartments in Indiranagar between 1000 sqft and 2000 sqft sorted by sqft low to high
    Output: {{"bhk": null, "location": "Indiranagar", "max_price": null, "min_price": null, "min_sqft": 1000.0, "max_sqft": 2000.0, "sort_by": "sqft", "sort_order": "asc"}}

    Query:
    List 4 BHK villas in Sarjapur Road above 2 crores
    Output: {{"bhk": 4, "location": "Sarjapur Road", "max_price": null, "min_price": 200.0, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null}}

    Query:
    Show me 1 BHK apartments in Electronic City under 50 lakhs sorted by price high to low
    Output: {{"bhk": 1, "location": "Electronic City", "max_price": 50.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": "price", "sort_order": "desc"}}

    Query:
    Find apartments in Hebbal with minimum 1200 sqft area
    Output: {{"bhk": null, "location": "Hebbal", "max_price": null, "min_price": null, "min_sqft": 1200.0, "max_sqft": null, "sort_by": null, "sort_order": null}}
    
    Query:
    Show me 2 BHK flats in Marathahalli between 60 lakhs and 90 lakhs
    Output: {{"bhk": 2, "location": "Marathahalli", "max_price": 90.0, "min_price": 60.0, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null}}

    Query:
    Show me 3 BHK apartments in Whitefield above 1 acre sorted by price high to low
    Output: {{"bhk": 3, "location": "Whitefield", "max_price": null, "min_price": null, "min_sqft": 43560.0, "max_sqft": null, "sort_by": "price", "sort_order": "desc"}}

    # 1 acre = 43560 sqft

    Query:
    Find 2 BHK flats in Koramangala between 100 sqm and 200 sqm
    Output: {{"bhk": 2, "location": "Koramangala", "max_price": null, "min_price": null, "min_sqft": 1076.0, "max_sqft": 2152.0, "sort_by": null, "sort_order": null}}

    # 1 sqm = 10.764 sqft

    Query:
    Show me villas in Indiranagar above 500 sq yards sorted by sqft low to high
    Output: {{"bhk": null, "location": "Indiranagar", "max_price": null, "min_price": null, "min_sqft": 4500.0, "max_sqft": null, "sort_by": "sqft", "sort_order": "asc"}}

    # 1 square yard = 9 sqft → 500 sq yards = 4,500 sqft

    Query: Show me 3 BHK apartments in Whitefield under 3 crores sorted by sqft high to low
    Output: {{"bhk": 3, "location": "Whitefield", "max_price": 300.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": "sqft", "sort_order": "desc"}}


    Now Process:
    Query: {query1}
    """ 


out1 = gen(prompt1, max_new_tokens=300, do_sample=False , max_length = 4096 , return_full_text = False)

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [62]:
print(out1[0]['generated_text'])

 Output: {"bhk": 4, "location": "Whitefield", "max_price": 100.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null}
    """
query = """Show me 4-BHK apartments in Whitefield with price less than 1 crore"""

# YOUR CODE GOES HERE
result = {
    "bhk": 4,
    "location": "Whitefield",
    "max_price": 100.0,
    "min_price": None,
    "min_sqft": None,
    "max_sqft": None,
    "sort_by": None,
    "sort_order": None
}


Cleaning the output of the LLM 

In [63]:
output = out1[0]['generated_text']
output

' Output: {"bhk": 4, "location": "Whitefield", "max_price": 100.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null}\n    """\nquery = """Show me 4-BHK apartments in Whitefield with price less than 1 crore"""\n\n# YOUR CODE GOES HERE\nresult = {\n    "bhk": 4,\n    "location": "Whitefield",\n    "max_price": 100.0,\n    "min_price": None,\n    "min_sqft": None,\n    "max_sqft": None,\n    "sort_by": None,\n    "sort_order": None\n}'

In [64]:
output = output.replace("\n" , " ")
output

' Output: {"bhk": 4, "location": "Whitefield", "max_price": 100.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null}     """ query = """Show me 4-BHK apartments in Whitefield with price less than 1 crore"""  # YOUR CODE GOES HERE result = {     "bhk": 4,     "location": "Whitefield",     "max_price": 100.0,     "min_price": None,     "min_sqft": None,     "max_sqft": None,     "sort_by": None,     "sort_order": None }'

In [65]:
output = output.split("result = ")[-1]
output

'{     "bhk": 4,     "location": "Whitefield",     "max_price": 100.0,     "min_price": None,     "min_sqft": None,     "max_sqft": None,     "sort_by": None,     "sort_order": None }'

In [67]:
output = output.replace("None" , "null")
output

'{     "bhk": 4,     "location": "Whitefield",     "max_price": 100.0,     "min_price": null,     "min_sqft": null,     "max_sqft": null,     "sort_by": null,     "sort_order": null }'

In [68]:
output.replace("    " , "")

'{ "bhk": 4, "location": "Whitefield", "max_price": 100.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null }'

In [36]:
import re

In [38]:
output

'{"bhk": 4, "location": "Whitefield", "max_price": 100.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null}'

In [69]:
# convert to python dict
import json
output_dict = json.loads(output)
output_dict

{'bhk': 4,
 'location': 'Whitefield',
 'max_price': 100.0,
 'min_price': None,
 'min_sqft': None,
 'max_sqft': None,
 'sort_by': None,
 'sort_order': None}

In [70]:
output_dict

{'bhk': 4,
 'location': 'Whitefield',
 'max_price': 100.0,
 'min_price': None,
 'min_sqft': None,
 'max_sqft': None,
 'sort_by': None,
 'sort_order': None}

In [ ]:
import json


def extract_filters_llm(query):

    prompt = f"""
    Extract filters from the real estate query.

    Return ONLY valid JSON with these fields:
    - bhk (int or null)
    - location (string or null)
    - max_price (float or null , in lakhs) 
    - min_price (float or null , in lakhs) 
    - min_sqft (float or null , in sqft)
    - max_sqft (float or null , in sqft) 
    - sort_by ("price" or "sqft" or ["price", "sqft"] or ["sqft", "price"])
    - sort_order ("asc" or "desc" or ["asc", "desc"] or ["desc", "asc"])

    INSTRUCTION:
    1. DO NOT RETURN ANY EXPLANATION. ONLY RETURN THE JSON.DO NOT RETURN ANY CODE FOR QUERY MANIPULATION TO JSON. ONLY RETURN THE JSON.

    2. sort_by can be a list if multiple sort criteria are mentioned in the query. 
    eg - ["price" , "total_sqft"] or ["total_sqft", "price"]

    3. sort_order can also be a list if multiple sort criteria are mentioned in the query. The order of sort_order should correspond to the order of sort_by.
    eg - ["asc", "desc"] or ["desc", "asc"]
    
    
    Examples:
    Query:
    Show me 2 BHK apartments in Whitefield under 80 lakhs sorted by price low to high
    Output: {{"bhk": 2, "location": "Whitefield", "max_price": 80.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": "price", "sort_order": "asc"}}

    Query:
    Find 3 BHK flats in Koramangala under 1.5 crores
    Output: {{"bhk": 3, "location": "Koramangala", "max_price": 150.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null}}

    Query:
    Show me apartments in Indiranagar between 1000 sqft and 2000 sqft sorted by sqft low to high
    Output: {{"bhk": null, "location": "Indiranagar", "max_price": null, "min_price": null, "min_sqft": 1000.0, "max_sqft": 2000.0, "sort_by": "sqft", "sort_order": "asc"}}

    Query:
    List 4 BHK villas in Sarjapur Road above 2 crores
    Output: {{"bhk": 4, "location": "Sarjapur Road", "max_price": null, "min_price": 200.0, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null}}

    Query:
    Show me 1 BHK apartments in Electronic City under 50 lakhs sorted by price high to low
    Output: {{"bhk": 1, "location": "Electronic City", "max_price": 50.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": "price", "sort_order": "desc"}}

    Query:
    Find apartments in Hebbal with minimum 1200 sqft area
    Output: {{"bhk": null, "location": "Hebbal", "max_price": null, "min_price": null, "min_sqft": 1200.0, "max_sqft": null, "sort_by": null, "sort_order": null}}
    
    Query:
    Show me 2 BHK flats in Marathahalli between 60 lakhs and 90 lakhs
    Output: {{"bhk": 2, "location": "Marathahalli", "max_price": 90.0, "min_price": 60.0, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null}}

    Query:
    Show me 3 BHK apartments in Whitefield above 1 acre sorted by price high to low
    Output: {{"bhk": 3, "location": "Whitefield", "max_price": null, "min_price": null, "min_sqft": 43560.0, "max_sqft": null, "sort_by": "price", "sort_order": "desc"}}

    # 1 acre = 43560 sqft

    Query:
    Find 2 BHK flats in Koramangala between 100 sqm and 200 sqm
    Output: {{"bhk": 2, "location": "Koramangala", "max_price": null, "min_price": null, "min_sqft": 1076.0, "max_sqft": 2152.0, "sort_by": null, "sort_order": null}}

    # 1 sqm = 10.764 sqft

    Query:
    Show me villas in Indiranagar above 500 sq yards sorted by sqft low to high
    Output: {{"bhk": null, "location": "Indiranagar", "max_price": null, "min_price": null, "min_sqft": 4500.0, "max_sqft": null, "sort_by": "sqft", "sort_order": "asc"}}

    # 1 square yard = 9 sqft → 500 sq yards = 4,500 sqft

    Query: Show me 3 BHK apartments in Whitefield under 3 crores sorted by sqft high to low
    Output: {{"bhk": 3, "location": "Whitefield", "max_price": 300.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": "sqft", "sort_order": "desc"}}

    Query:
    Show me 2 BHK apartments in Whitefield sorted by price low to high and sqft high to low
    Output: {{
    "bhk": 2,
    "location": "Whitefield",
    "max_price": null,
    "min_price": null,
    "min_sqft": null,
    "max_sqft": null,
    "sort_by": ["price", "total_sqft"],
    "sort_order": ["asc", "desc"]
    }}

    Query:
    List 3 BHK flats in Koramangala under 2 crores sorted first by sqft low to high then by price high to low
    Output: {{
    "bhk": 3,
    "location": "Koramangala",
    "max_price": 200.0,
    "min_price": null,
    "min_sqft": null,
    "max_sqft": null,
    "sort_by": ["total_sqft", "price"],
    "sort_order": ["asc", "desc"]
    }}

    Query:
    Find apartments in Indiranagar between 1000 sqft and 2000 sqft sorted by price high to low and sqft low to high
    Output: {{
    "bhk": null,
    "location": "Indiranagar",
    "max_price": null,
    "min_price": null,
    "min_sqft": 1000.0,
    "max_sqft": 2000.0,
    "sort_by": ["price", "total_sqft"],
    "sort_order": ["desc", "asc"]
    }}


    Now Process:
    Query: {query}
    """ 
    out = gen(prompt , max_new_tokens = 200 , do_sample = False ,return_full_text = False)

    text = out[0]['generated_text']

    # # For debugging purpose
    # print("LLM filters Extraction Output:\n", text)


    try:
        match = text.replace("\n" , " ")
        filters = match.split("result = ")[-1]
        filters = filters.replace("None" , "null")
        
        filters_dict = json.loads(filters)
        return filters_dict
    except Exception as e:
        print("Error in extracting filters:", e)
        return None

# This functions returns filters in python dictionary format

In [41]:
# Testing the extract_filters_llm function by giving a sample query

query = "Show me 2 BHK apartments in Whitefield with price less than 80 lakhs"
filters = extract_filters_llm(query)
print(filters)

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LLM filters Extraction Output:  Output: {"bhk": 2, "location": "Whitefield", "max_price": 80.0, "min_price": null, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null}

    Query: Show me 3 BHK flats in Koramangala with price more than 1.5 crores
    Output: {"bhk": 3, "location": "Koramangala", "max_price": null, "min_price": 150.0, "min_sqft": null, "max_sqft": null, "sort_by": null, "sort_order": null}

    Query: Show me apartments in Indiranagar with total sqft greater than 1000 sqft
    Output: {"bhk": null, "location": "Indiranagar", "max_price": null, "min_price": null, "
{'bhk': 2, 'location': 'Whitefield', 'max_price': 80.0, 'min_price': None, 'min_sqft': None, 'max_sqft': None, 'sort_by': None, 'sort_order': None}


In [43]:
# testing the multi sort criteria
query1 = "Show me 2 BHK apartments in Whitefield sorted by price low to high and sqft high to low"
filters1 = extract_filters_llm(query1)
print(filters1)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'bhk': 2, 'location': 'Whitefield', 'max_price': None, 'min_price': None, 'min_sqft': None, 'max_sqft': None, 'sort_by': ['price', 'total_sqft'], 'sort_order': ['asc', 'desc']}


Filter Function

In [72]:
def apply_filters(df , filters):

    result = df.copy()

    if filters.get('bhk'):
        result = result[result['bhk'] == filters['bhk']]

    if filters.get('location'):
        result = result[result['location'].str.lower() == filters['location'].lower()]
    if filters.get('max_price'):
        result = result[result['price'] <= filters['max_price']] 

    if filters.get('min_price'):
        result = result[result['price'] >= filters['min_price']]

    if filters.get('min_sqft'):
        result = result[result['total_sqft'] >= filters['min_sqft']]

    if filters.get('max_sqft'):
        result = result[result['total_sqft'] <= filters['max_sqft']]

    return result 



In [50]:
df.shape, df.columns

((7129, 10),
 Index(['area_type', 'availability', 'location', 'size', 'society',
        'total_sqft', 'bath', 'balcony', 'price', 'bhk'],
       dtype='object'))

In [51]:
output_dict

{'bhk': 4,
 'location': 'Whitefield',
 'max_price': 100.0,
 'min_price': None,
 'min_sqft': None,
 'max_sqft': None,
 'sort_by': None,
 'sort_order': None}

In [53]:
filtered_df = apply_filters(df , output_dict)
filtered_df

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk
5690,Built-up Area,18-Aug,Whitefield,4 Bedroom,M1riaa,2500.0,4.0,0.0,100.0,4


Sorting : Multi variable sorting

In [73]:
def apply_sorting(df, filters):
    sort_by = filters.get('sort_by')
    order = filters.get("sort_order" , 'asc')

    # default 
    ascending = True if order == 'asc' else False 

    # MULTI SORTING SUPPORT:
    if isinstance(sort_by , list):
        ascending_list = []
        for col, ord in zip(sort_by , filters.get('sort_order',[])):
            ascending_list.append(True if ord == 'asc' else False)
        df = df.sort_values(by= sort_by , ascending = ascending_list)

    else:
        if sort_by in ['price', 'total_sqft']:
            df = df.sort_values(by=sort_by, ascending=ascending)

    return df

In [55]:
apply_sorting(filtered_df, output_dict)

,area_type,availability,location,size,society,total_sqft,bath,balcony,price,bhk
5690,Built-up Area,18-Aug,Whitefield,4 Bedroom,M1riaa,2500.0,4.0,0.0,100.0,4


In [ ]:
# # Trying Memory Implementation 
# memory = {
#     'history' : [],
#     'last_results' : None
# }

# # Function to detect follow up queries 

# def is_followup(query):
#     query = query.lower()
#     keywords = ['first' , 'second' , 'third' , 'more' , 'similar' , 'next' , 'details' , "additional"]


#     return any(k in query for k in keywords)

In [ ]:
# Normal answer query function with memory implementation


# def answer_query(query , top_k = 5):

#     # Follow up query handling

#     if is_followup(query) and memory['last_results'] is not None:
#         # use the last results from memory to answer the follow up query
#         top_df = memory['last_results']

#         # detect index 

#         idx = 0 
#         if 'second'

In [ ]:
# Normal answer query function without memory implementation

def answer_query(query , top_k = 5):
    filters = extract_filters_llm(query)

    filtered = apply_filters(df , filters)

    sorted_df = apply_sorting(filtered , filters)

    top_df = sorted_df.head(top_k)

   
    context = "\n".join([
        f"""
Properties {i+1}:
- BHK: {row['bhk']}
- Location: {row['location']}
- Price: {row['price']} lakh
- Size : {row['total_sqft']} sqft
- Bathrooms: {row['bath']}
- Balcony: {row['balcony']}
- Area Type: {row['area_type']}
- Availability: {row['availability']}
- Society: {row['society']}
"""
for i , (_ , row) in enumerate(top_df.iterrows())
])
    

    prompt = f"""
    
    You are a real estate assistant.

    Use ONLY the given properties.

    Properties:
    {context}

    User Query: {query}

    Answer in bullet points:

    """

    out = gen(prompt , max_new_tokens = 200 , do_sample = False , return_full_text = False)

    return out[0]['generated_text']
    

In [75]:
query = "Show me 2 BHK apartments in Whitefield under 80 lakhs sorted by price low to high"

answer = answer_query(query)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LLM filters Extraction Output:
  """

query = "Show me 2 BHK apartments in Whitefield under 80 lakhs sorted by price low to high"

# YOUR CODE HERE
result = {
    "bhk": 2,
    "location": "Whitefield",
    "max_price": 80.0,
    "min_price": None,
    "min_sqft": None,
    "max_sqft": None,
    "sort_by": "price",
    "sort_order": "asc"
}


In [76]:
print(answer)


    
    You are a real estate assistant.

    Use ONLY the given properties.

    Properties:
    
Properties 1:
- BHK: 2
- Location: Whitefield
- Price: 32.0 lakh
- Size : 1024.0 sqft
- Bathrooms: 2.0
- Balcony: 1.0
- Area Type: Super built-up  Area
- Availability: Ready To Move
- Society: Savarar


Properties 2:
- BHK: 2
- Location: Whitefield
- Price: 34.555 lakh
- Size : 1115.0 sqft
- Bathrooms: 2.0
- Balcony: 0.0
- Area Type: Super built-up  Area
- Availability: 20-Dec
- Society: Somns T


Properties 3:
- BHK: 2
- Location: Whitefield
- Price: 35.0 lakh
- Size : 805.0 sqft
- Bathrooms: 2.0
- Balcony: 2.0
- Area Type: Super built-up  Area
- Availability: Ready To Move
- Society: JahanSa


Properties 4:
- BHK: 2
- Location: Whitefield
- Price: 35.0 lakh
- Size : 735.0 sqft
- Bathrooms: 2.0
- Balcony: 1.0
- Area Type: Built-up  Area
- Availability: Ready To Move
- Society: JahanSa


Properties 5:
- BHK: 2
- Location: Whitefield
- Price: 35.0 lakh
- Size : 1095.0 sqft
- Bathrooms: 2